In [1]:
import os
import pandas as pd
import numpy as np
import shutil
import sys
import tqdm.notebook as tq
from collections import defaultdict

import torch
import torch.nn as nn

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [2]:
import pandas as pd 
data_os = pd.read_csv("OurData/os.csv")
data_qt = pd.read_csv("OurData/qt.csv")
data_android = pd.read_csv("OurData/android.csv")

In [3]:
train_data = pd.concat([data_android, data_qt], ignore_index=True)
test_data = data_os

In [ ]:
train_data

In [ ]:
train_data = train_data.drop(columns=['Label','others'])

In [ ]:
test_data = test_data.drop(columns=['Label'])

In [5]:
train_data["combined"] = train_data["Title"] + ". " + train_data["Description"]+". "+train_data["ChangedFiles"]+". "+train_data["files_diff_dict"]
test_data["combined"] = test_data["Title"] + ". " + test_data["Description"]+". "+train_data["ChangedFiles"]+". "+train_data["files_diff_dict"]


In [6]:
from sklearn.model_selection import train_test_split
train_data=train_data.drop(columns=['OwnerName','Title','Description','ChangedFiles','files_diff_dict','Id'])
train_data, df_valid = train_test_split(train_data, random_state=88, test_size=0.8, shuffle=True)

In [7]:
test_data=test_data.drop(columns=['OwnerName','Title','Description','ChangedFiles','files_diff_dict','Id'])

In [8]:
# Hyperparameters
MAX_LEN = 256
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32
TEST_BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-05
THRESHOLD = 0.5 # threshold for the sigmoid

In [9]:
from transformers import BertTokenizer, BertModel

In [10]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

In [ ]:
# Test the tokenizer
test_text = "We are testing BERT tokenizer."
# generate encodings
encodings = tokenizer.encode_plus(test_text, 
                                  add_special_tokens = True,
                                  max_length = 50,
                                  truncation = True,
                                  padding = "max_length", 
                                  return_attention_mask = True, 
                                  return_tensors = "pt")
# we get a dictionary with three keys (see: https://huggingface.co/transformers/glossary.html) 
encodings

In [ ]:
train_data['combined']


In [13]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len, target_list):
        self.tokenizer = tokenizer
        self.df = df
        self.title = list(df['combined'])
        self.targets = self.df[target_list].values
        self.max_len = max_len

    def __len__(self):
        return len(self.title)

    def __getitem__(self, index):
        title = str(self.title[index])
        title = " ".join(title.split())
        inputs = self.tokenizer.encode_plus(
            title,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=True,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'token_type_ids': inputs["token_type_ids"].flatten(),
            'targets': torch.FloatTensor(self.targets[index]),
            'title': title
        }

In [14]:
target_list = ["bug","test","feature","resource","deprecat","merge","refactor"]
df_train = train_data
df_test = test_data


In [15]:
train_dataset = CustomDataset(df_train, tokenizer, MAX_LEN, target_list)
valid_dataset = CustomDataset(df_valid, tokenizer, MAX_LEN, target_list)
test_dataset = CustomDataset(df_test, tokenizer, MAX_LEN, target_list)

In [ ]:
next(iter(train_dataset))


In [17]:
train_data_loader = torch.utils.data.DataLoader(train_dataset, 
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_data_loader = torch.utils.data.DataLoader(valid_dataset, 
    batch_size=VALID_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_data_loader = torch.utils.data.DataLoader(test_dataset, 
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [18]:
import torch
from transformers import AutoModel



class BERTClass(torch.nn.Module):
    def __init__(self):
        super(BERTClass, self).__init__()
        self.roberta = AutoModel.from_pretrained('roberta-base')
#         self.l2 = torch.nn.Dropout(0.3)
        self.fc = torch.nn.Linear(768,7)
    
    def forward(self, ids, mask, token_type_ids):
        _, features = self.roberta(ids, attention_mask = mask, token_type_ids = token_type_ids, return_dict=False)
#         output_2 = self.l2(output_1)
        output = self.fc(features)
        return output

model = BERTClass()
model.to(device);


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [ ]:
from transformers import AdamW

# define the optimizer
optimizer = AdamW(model.parameters(), lr = 1e-5)         

In [21]:
# Training of the model for one epoch
def train_model(training_loader, model, optimizer):

    losses = []
    correct_predictions = 0
    num_samples = 0
    # set model to training mode (activate droput, batch norm)
    model.train()
    # initialize the progress bar
    loop = tq.tqdm(enumerate(training_loader), total=len(training_loader), 
                      leave=True, colour='steelblue')
    for batch_idx, data in loop:
        ids = data['input_ids'].to(device, dtype = torch.long)
        mask = data['attention_mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        # forward
        outputs = model(ids, mask, token_type_ids) # (batch,predict)=(32,8)
        loss = loss_fn(outputs, targets)
        losses.append(loss.item())
        # training accuracy, apply sigmoid, round (apply thresh 0.5)
        outputs = torch.sigmoid(outputs).cpu().detach().numpy().round()
        targets = targets.cpu().detach().numpy()
        correct_predictions += np.sum(outputs==targets)
        num_samples += targets.size   # total number of elements in the 2D array

        # backward
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # grad descent step
        optimizer.step()

        # Update progress bar
        #loop.set_description(f"")
        #loop.set_postfix(batch_loss=loss)

    # returning: trained model, model accuracy, mean loss
    return model, float(correct_predictions)/num_samples, np.mean(losses)

In [23]:
def eval_model(validation_loader, model, optimizer):
    losses = []
    correct_predictions = 0
    num_samples = 0
    # set model to eval mode (turn off dropout, fix batch norm)
    model.eval()

    with torch.no_grad():
        for batch_idx, data in enumerate(validation_loader, 0):
            ids = data['input_ids'].to(device, dtype = torch.long)
            mask = data['attention_mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = model(ids, mask, token_type_ids)

            loss = loss_fn(outputs, targets)
            losses.append(loss.item())

            # validation accuracy
            # add sigmoid, for the training sigmoid is in BCEWithLogitsLoss
            outputs = torch.sigmoid(outputs).cpu().detach().numpy().round()
            targets = targets.cpu().detach().numpy()
            correct_predictions += np.sum(outputs==targets)
            num_samples += targets.size   # total number of elements in the 2D array

    return float(correct_predictions)/num_samples, np.mean(losses)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Define your metric calculations
def compute_metrics(outputs, targets):
    outputs = torch.sigmoid(outputs).cpu().detach().numpy().round()
    targets = targets.cpu().detach().numpy()
    accuracy = np.sum(outputs == targets) / targets.size
    f1 = f1_score(targets, outputs, average='macro')
    precision = precision_score(targets, outputs, average='macro')
    recall = recall_score(targets, outputs, average='macro')
    return accuracy, f1, precision, recall

history = defaultdict(list)
best_accuracy = 0

for epoch in range(1, EPOCHS + 1):
    print(f'Epoch {epoch}/{EPOCHS}')
    model, train_acc, train_loss = train_model(train_data_loader, model, optimizer)
    val_acc, val_loss = eval_model(val_data_loader, model, optimizer)

    # Compute validation metrics
    val_preds, val_targets = [], []
    model.eval()
    with torch.no_grad():
        for batch in val_data_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            targets = batch['targets'].to(device)
            outputs = model(ids, mask, token_type_ids)
            val_preds.append(outputs.cpu())
            val_targets.append(targets.cpu())

    val_preds = torch.cat(val_preds)
    val_targets = torch.cat(val_targets)
    val_acc, val_f1, val_precision, val_recall = compute_metrics(val_preds, val_targets)

    print(f'train_loss={train_loss:.4f}, val_loss={val_loss:.4f} train_acc={train_acc:.4f}, val_acc={val_acc:.4f}')
    print(f'val_f1={val_f1:.4f}, val_precision={val_precision:.4f}, val_recall={val_recall:.4f}')

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)

    # Save the best model
    if val_acc > best_accuracy:
        torch.save(model.state_dict(), os.path.join('./output', "MLTC_model_state.bin"))
        best_accuracy = val_acc


In [26]:
import matplotlib.pyplot as plt

In [ ]:

plt.rcParams["figure.figsize"] = (10,7)
plt.plot(history['train_acc'], label='train accuracy')
plt.plot(history['val_acc'], label='validation accuracy')
plt.title('Training history')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.ylim([0, 1]);
plt.grid()

In [ ]:
model = BERTClass()
model.load_state_dict(torch.load(os.path.join('./',"output","MLTC_model_state.bin")))
model = model.to(device)

In [27]:
test_acc, test_loss = eval_model(test_data_loader, model, optimizer)

In [ ]:
test_acc

In [29]:
from sklearn.metrics import confusion_matrix, classification_report

In [30]:
def get_predictions(model, data_loader):
    """
    Outputs:
      predictions - 
    """
    model = model.eval()
    
    titles = []
    predictions = []
    prediction_probs = []
    target_values = []

    with torch.no_grad():
      for data in data_loader:
        title = data["title"]
        ids = data["input_ids"].to(device, dtype = torch.long)
        mask = data["attention_mask"].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data["targets"].to(device, dtype = torch.float)
        
        outputs = model(ids, mask, token_type_ids)
        # add sigmoid, for the training sigmoid is in BCEWithLogitsLoss
        outputs = torch.sigmoid(outputs).detach().cpu()
        # thresholding at 0.5
        preds = outputs.round()
        targets = targets.detach().cpu()

        titles.extend(title)
        predictions.extend(preds)
        prediction_probs.extend(outputs)
        target_values.extend(targets)
    
    predictions = torch.stack(predictions)
    prediction_probs = torch.stack(prediction_probs)
    target_values = torch.stack(target_values)
    
    return titles, predictions, prediction_probs, target_values

In [60]:
titles, predictions, prediction_probs, target_values = get_predictions(model, test_data_loader)

In [39]:
from sklearn.metrics import classification_report, roc_auc_score, matthews_corrcoef
auc_roc = roc_auc_score(target_values, prediction_probs)

In [ ]:
auc_roc

In [ ]:
print(f"titles:{len(titles)} \npredictions:{predictions.shape} \nprediction_probs:{prediction_probs.shape} \ntarget_values:{target_values.shape}")

In [ ]:
report = classification_report(target_values, predictions, target_names=target_list, output_dict=True)

# Convert the classification report to a DataFrame
report_df = pd.DataFrame(report).transpose()

# Export the DataFrame to a CSV file
report_df.to_csv('classification_report_bert_WithChaF.csv', index=True)

In [ ]:
target_values 

In [ ]:
predictions

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_auc_score, matthews_corrcoef
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np

# Assuming target_values and predictions are lists of lists (for multi-label classification)
target_list = ["bug", "test", "feature", "resource", "deprecat", "merge", "refactor"]

# Convert to binary format for multi-label classification
mlb = MultiLabelBinarizer(classes=target_list)
target_binarized = mlb.fit_transform(target_values)
predictions_binarized = mlb.transform(predictions)

# Calculate precision, recall, f1-score
precision, recall, f1, _ = precision_recall_fscore_support(target_binarized, predictions_binarized, average=None)

# Calculate accuracy
accuracy = accuracy_score(target_binarized, predictions_binarized)

# Calculate AUC-ROC for each class
auc_scores = {}
for i, class_name in enumerate(target_list):
    if len(np.unique(target_binarized[:, i])) > 1:
        auc_scores[class_name] = roc_auc_score(target_binarized[:, i], predictions_binarized[:, i])
    else:
        auc_scores[class_name] = np.nan  # Use NaN for undefined AUC-ROC

# Calculate macro average AUC-ROC, ignoring NaN values
macro_auc = np.nanmean(list(auc_scores.values()))

# Add AUC-ROC to the report
for class_name in target_list:
    report[class_name]['auc'] = auc_scores[class_name]

# Add macro average AUC-ROC to the report
report['macro avg']['auc'] = macro_auc

# Calculate MCC for each class (binary one-vs-rest)
mcc_scores = {}
for i, class_name in enumerate(target_list):
    if len(np.unique(target_binarized[:, i])) > 1:
        mcc_scores[class_name] = matthews_corrcoef(target_binarized[:, i], predictions_binarized[:, i])
    else:
        mcc_scores[class_name] = np.nan  # Use NaN for undefined MCC

# Calculate macro average MCC, ignoring NaN values
macro_mcc = np.nanmean(list(mcc_scores.values()))

# Add MCC to the report
for class_name in target_list:
    report[class_name]['mcc'] = mcc_scores[class_name]

# Add macro average MCC to the report
report['macro avg']['mcc'] = macro_mcc

# Calculate micro average AUC-ROC and MCC
micro_auc = roc_auc_score(target_binarized, predictions_binarized, average='micro')
micro_mcc = matthews_corrcoef(target_binarized.ravel(), predictions_binarized.ravel())

report['micro avg']['auc'] = micro_auc
report['micro avg']['mcc'] = micro_mcc

# Create the report
report = {
    "label": target_list + ["macro avg", "micro avg"],
    "precision": list(precision) + [macro_precision, micro_precision],
    "recall": list(recall) + [macro_recall, micro_recall],
    "f1-score": list(f1) + [macro_f1, micro_f1],
    "accuracy": [accuracy] * (len(target_list) + 2),  # same accuracy for all as it's a global measure
    "auc": list(auc_scores.values()) + [macro_auc, micro_auc],
    "mcc": list(mcc_scores.values()) + [macro_mcc, micro_mcc]
}

# Convert report to DataFrame for a nicer display
import pandas as pd
report_df = pd.DataFrame(report)
report_df.set_index('label', inplace=True)

print(report_df)
